# 06. Capstone: Random Forest versus skeleton JEPA

We will fit both approaches on the same retained gait windows, select settings on validation sources, and evaluate prespecified methods on held-out sequences.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

In [ ]:
from aaai_gait.workflow import prepare_corpus
x, meta, provenance = prepare_corpus(settings)
print('Window tensor:', x.shape, '[windows, frames, selected joints, XY]')
display(meta.groupby(['split','label']).agg(windows=('sequence_id','size'), sequences=('sequence_id','nunique'), sources=('source_id','nunique')))

## 1. Rebuild the classical baseline on this cohort

The exp5 notebooks motivate a Random Forest on gait-angle summaries. Here we recompute all features from the exact same windows used by JEPA. We do not reuse exp5's saved feature table or historical scores because those involve different data and protocols.

The baseline contains 36 measured values: 15 core summaries of left/right hip, knee, and ankle angles; six angle standard deviations; 12 wrist/ankle motion summaries; and three ankle-separation/torso-orientation summaries. The optional `build_ambient_features` adapter reproduces the 15-field ambient core-angle vector. We do not advertise an 82-feature vector populated with uncomputed zeros.

![Comparison](images/evaluation.svg)

In [ ]:
from aaai_gait.features import build_features
features,feature_names=build_features(x,settings.target_fps)
print('Feature matrix:',features.shape)
display(pd.DataFrame({'feature':feature_names}))

## 2. Keep fitting and selection separate

The Random Forest pipeline fits its median imputer on training data only. It compares maximum tree depths of 4, 8, and unrestricted on validation macro-F1, using 300 trees and a fixed seed. The JEPA probes compare regularization strengths 0.1, 1, and 10. Every method sees the same train/validation/test source membership.

The condition task includes all five labels during supervised readout fitting, including the normal-only encoder's probe. Normal-only describes encoder pretraining, not an unsupervised five-way classifier. The separate anomaly task in notebook 05 uses no abnormal labels to learn its reference or threshold.

We also compare an untrained encoder, mean pose, pose availability with camera view, and a training-prior baseline. These controls test whether learned dynamics add value beyond architecture, static posture, and class frequency.

In [ ]:
from aaai_gait.workflow import capstone
result=capstone(x,meta,provenance,settings,bootstrap_repeats=200 if settings.demo else 1000)
rows=[]
for method,record in result['methods'].items():
    rows.append({'method':method,**{k:record['source_weighted'][k] for k in ['macro_f1','balanced_accuracy','abnormal_auroc','abnormal_average_precision']}})
display(pd.DataFrame(rows))
print('JEPA selected using validation:',result['selected_jepa'])
print('Interpretation:',result['interpretation'])

## 3. Read paired uncertainty correctly

Average overlapping-window probabilities inside each annotated sequence. Report both sequence-weighted and source-weighted metrics. The latter gives each source equal total weight across its sequences.

The bootstrap resamples complete source groups, using exactly the same sampled groups for both methods in a paired comparison. Its reported differences are sequence-weighted. Replicates missing a prespecified class are counted and omitted. With few test sources, many replicates may be invalid and intervals may be unavailable. These intervals do not include training-seed uncertainty, and source groups do not prove participant independence.

In [ ]:
display(result['paired_selected_minus_rf_sequence_weighted'])
chosen=result['selected_jepa']
record=result['methods'][chosen]['source_weighted']
confusion=np.asarray(record['confusion_matrix'])
fig,ax=plt.subplots(figsize=(7,5));image=ax.imshow(confusion,cmap='Blues')
ax.set_xticks(range(5),LABELS,rotation=35,ha='right');ax.set_yticks(range(5),LABELS)
ax.set_xlabel('Predicted label');ax.set_ylabel('Dataset label');ax.set_title('Synthetic software check' if settings.demo else 'Source-weighted test confusion')
for i in range(5):
    for j in range(5):ax.text(j,i,f'{confusion[i,j]:.1f}',ha='center',va='center')
fig.colorbar(image);fig.tight_layout();plt.show()

## 4. Predict one held-out gait sequence

A video must pass the same annotation-aware pose extraction, temporal resampling, and QC before either classifier receives it. The example below fits a frozen probe using training rows, then predicts all windows of one test sequence and averages their probabilities. The Random Forest follows the identical interface using `build_features`. For a new unannotated video, a human-reviewed person crop and frame interval must first be supplied. Do not classify a random bystander or silently assume a new video belongs to a known condition.

In [ ]:
from aaai_gait.workflow import normal_anchor, progressive_model, select_probe
from aaai_gait.training import embed_windows
anchor,_=normal_anchor(x,meta,provenance,settings)
encoder=anchor if chosen=='normal_jepa' else progressive_model(anchor,x,meta,provenance,settings)[0]
z=embed_windows(encoder,x,batch_size=settings.batch_size)
probe,_=select_probe(z,meta,settings.seed)
sequence_id=meta.loc[meta.split.eq('test'),'sequence_id'].iloc[0]
sequence_rows=meta.sequence_id.eq(sequence_id).to_numpy()
from aaai_gait.media import show_window_context
show_window_context(meta.loc[sequence_rows].iloc[0],manifest)
p=probe.predict_proba(z[sequence_rows]).mean(0)
display(pd.Series(p,index=probe.classes_,name=f'Observed sequence {sequence_id}'))

## 5. Run multiple prespecified splits

Use a fixed list of seeds chosen before examining results. Each run retrains normal and progressive encoders from scratch within its training partition. Reusing an encoder trained on all sources would leak test motion even without labels. Repeated split scores are dependent because sources recur. Report the distribution across seeds and keep source-cluster intervals within each run.

```bash
uv run python scripts/run_pipeline.py train --seeds 42 43 44
```

The results are an internal GAVD comparison, not a direct numerical reproduction of exp5 and not evidence of diagnostic accuracy.

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.